In [1]:
import sys
import os
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import preprocess_normed_images
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
device = "cuda"
dtype = torch.bfloat16

# Import Tokenizer & Model
---

In [3]:
processor = AutoProcessor.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth")

[transformers] The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [4]:
vlm = AutoModelForImageTextToText.from_pretrained(
    "unsloth/Qwen3.5-4B",
    dtype = torch.bfloat16,
    device_map = "cuda"
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [5]:
image = Image.open("/home/ubuntu/Shree_FYP/train/stage2/test/images/cat.png").resize((1024, 1024))
image2 = Image.open("/home/ubuntu/Shree_FYP/train/stage2/test/images/dog.png").resize((1024, 1024))

In [6]:
messages = [
    [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Describe this image."}
            ]
        }
    ],
    [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image2},
                {"type": "text", "text": "What breed is this dog?"}
            ]
        }
    ]
    
]

In [7]:
prompt = processor.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_dict = True,
    padding = True,
    return_tensors = "pt",
).to(vlm.device)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [8]:
with torch.no_grad():
    output = vlm(
        **prompt,
        output_hidden_states = True)

In [9]:
def extract_vision_hidden(
    model,
    processor,
    prompt,
    layer_idx=24,
    device=device,
    stack_if_possible=True,
):
    prompt = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in prompt.items()
    }

    with torch.no_grad():
        output = model(
            **prompt,
            output_hidden_states=True,
            use_cache=False,
        )

    hidden = output.hidden_states[layer_idx]
    input_ids = prompt["input_ids"]

    image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    image_mask = input_ids == image_token_id

    vision_hidden_list = [
        hidden[b, image_mask[b], :]
        for b in range(hidden.shape[0])
    ]

    image_lengths = [x.shape[0] for x in vision_hidden_list]

    if stack_if_possible and len(set(image_lengths)) == 1:
        vision_hidden = torch.stack(vision_hidden_list, dim=0)
        return vision_hidden, output

    return vision_hidden_list, output

In [10]:
vision_hidden, output = extract_vision_hidden(
    model=vlm,
    processor=processor,
    prompt=prompt,
    layer_idx=24,
    device="cuda",
)

if isinstance(vision_hidden, list):
    print("Different image token lengths:")
    for i, x in enumerate(vision_hidden):
        print(i, x.shape)
else:
    print("Stacked vision hidden:", vision_hidden.shape)

Stacked vision hidden: torch.Size([2, 1024, 2560])


In [11]:
vggt_model = VGGT(
    enable_camera=False,
    enable_point=False,
    enable_depth=False,
    enable_track=False,
    feature_only=True
).to(device)

vggt_model.eval()

vggt_model.load_state_dict(torch.load("/home/ubuntu/Shree_FYP/model.pt"), strict = False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['camera_head.empty_pose_tokens', 'camera_head.trunk.0.norm1.weight', 'camera_head.trunk.0.norm1.bias', 'camera_head.trunk.0.attn.qkv.weight', 'camera_head.trunk.0.attn.qkv.bias', 'camera_head.trunk.0.attn.proj.weight', 'camera_head.trunk.0.attn.proj.bias', 'camera_head.trunk.0.ls1.gamma', 'camera_head.trunk.0.norm2.weight', 'camera_head.trunk.0.norm2.bias', 'camera_head.trunk.0.mlp.fc1.weight', 'camera_head.trunk.0.mlp.fc1.bias', 'camera_head.trunk.0.mlp.fc2.weight', 'camera_head.trunk.0.mlp.fc2.bias', 'camera_head.trunk.0.ls2.gamma', 'camera_head.trunk.1.norm1.weight', 'camera_head.trunk.1.norm1.bias', 'camera_head.trunk.1.attn.qkv.weight', 'camera_head.trunk.1.attn.qkv.bias', 'camera_head.trunk.1.attn.proj.weight', 'camera_head.trunk.1.attn.proj.bias', 'camera_head.trunk.1.ls1.gamma', 'camera_head.trunk.1.norm2.weight', 'camera_head.trunk.1.norm2.bias', 'camera_head.trunk.1.mlp.fc1.weight', 'camera_head.trunk.1.mlp.fc1.bias', 'camer

In [12]:
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision.transforms import functional as TVF


def preprocess_raw_images_for_vggt(
    image_files,
    mode="crop",
    target_size=518,
    device=None,
    dtype=torch.float32,
):
    """
    Preprocess raw image files for VGGT.

    Args:
        image_files:
            List of image paths for one sample.
            Example: [img1_path, img2_path]

        mode:
            "crop" or "pad"

    Returns:
        Tensor with shape [1, N, 3, 518, 518]
        where N = number of input images.
    """

    if mode not in ["crop", "pad"]:
        raise ValueError("mode must be either 'crop' or 'pad'")

    processed_images = []

    for image_file in image_files:
        # Load image as RGB
        image = Image.open(image_file).convert("RGB")

        # Convert to tensor in [0, 1], shape [3, H, W]
        image = TVF.to_tensor(image).to(dtype=dtype)

        if device is not None:
            image = image.to(device)

        C, H, W = image.shape

        if mode == "crop":
            # Resize so the shorter side becomes 518, then center crop
            scale = target_size / min(H, W)

            new_h = round(H * scale / 14) * 14
            new_w = round(W * scale / 14) * 14

            # Make sure both sides are at least 518
            new_h = max(new_h, target_size)
            new_w = max(new_w, target_size)

            image = F.interpolate(
                image.unsqueeze(0),
                size=(new_h, new_w),
                mode="bicubic",
                align_corners=False,
            ).squeeze(0)

            top = (new_h - target_size) // 2
            left = (new_w - target_size) // 2

            image = image[
                :,
                top : top + target_size,
                left : left + target_size,
            ]

        else:
            # Resize so the longer side becomes 518, then pad
            scale = target_size / max(H, W)

            new_h = round(H * scale / 14) * 14
            new_w = round(W * scale / 14) * 14

            new_h = max(new_h, 14)
            new_w = max(new_w, 14)

            image = F.interpolate(
                image.unsqueeze(0),
                size=(new_h, new_w),
                mode="bicubic",
                align_corners=False,
            ).squeeze(0)

            pad_h = target_size - new_h
            pad_w = target_size - new_w

            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left

            image = F.pad(
                image,
                pad=(pad_left, pad_right, pad_top, pad_bottom),
                mode="constant",
                value=1.0,  # white padding
            )

        processed_images.append(image)

    # [N, 3, 518, 518]
    images = torch.stack(processed_images, dim=0)

    # Add batch dimension: [1, N, 3, 518, 518]
    images = images.unsqueeze(0)

    return images

In [13]:
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision.transforms import functional as TVF


def preprocess_raw_image_tensor_for_vggt(
    image,
    mode="crop",
    target_size=518,
    dtype=torch.float32,
    device=None,
):
    """
    image: PIL image
    returns: [3, 518, 518]
    """

    if mode not in ["crop", "pad"]:
        raise ValueError("mode must be either 'crop' or 'pad'")

    image = image.convert("RGB")
    image = TVF.to_tensor(image).to(dtype=dtype)

    if device is not None:
        image = image.to(device)

    C, H, W = image.shape

    if mode == "crop":
        scale = target_size / min(H, W)

        new_h = round(H * scale / 14) * 14
        new_w = round(W * scale / 14) * 14

        new_h = max(new_h, target_size)
        new_w = max(new_w, target_size)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(new_h, new_w),
            mode="bicubic",
            align_corners=False,
        ).squeeze(0)

        top = (new_h - target_size) // 2
        left = (new_w - target_size) // 2

        image = image[:, top:top + target_size, left:left + target_size]

    else:
        scale = target_size / max(H, W)

        new_h = round(H * scale / 14) * 14
        new_w = round(W * scale / 14) * 14

        new_h = max(new_h, 14)
        new_w = max(new_w, 14)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(new_h, new_w),
            mode="bicubic",
            align_corners=False,
        ).squeeze(0)

        pad_h = target_size - new_h
        pad_w = target_size - new_w

        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left

        image = F.pad(
            image,
            pad=(pad_left, pad_right, pad_top, pad_bottom),
            mode="constant",
            value=1.0,
        )

    return image


def preprocess_raw_image_batches_for_vggt(
    image_batches,
    mode="crop",
    target_size=518,
    dtype=torch.float32,
    device=None,
):
    """
    image_batches:
        List of samples.
        
        For batch size 2, one image each:
            [
                ["img1.png"],
                ["img2.png"],
            ]

        For batch size 1, two images/views:
            [
                ["img1.png", "img2.png"],
            ]

        For batch size 2, two images/views each:
            [
                ["sample0_front.png", "sample0_wrist.png"],
                ["sample1_front.png", "sample1_wrist.png"],
            ]

    returns:
        [B, N, 3, 518, 518]
    """

    num_images_per_sample = [len(sample) for sample in image_batches]

    if len(set(num_images_per_sample)) != 1:
        raise ValueError(
            f"All samples must have the same number of images for tensor batching. "
            f"Got num_images_per_sample={num_images_per_sample}"
        )

    batch_images = []

    for sample_image_files in image_batches:
        sample_images = []

        for image_file in sample_image_files:
            image = Image.open(image_file)

            image = preprocess_raw_image_tensor_for_vggt(
                image,
                mode=mode,
                target_size=target_size,
                dtype=dtype,
                device=device,
            )

            sample_images.append(image)

        # [N, 3, 518, 518]
        sample_images = torch.stack(sample_images, dim=0)
        batch_images.append(sample_images)

    # [B, N, 3, 518, 518]
    batch_images = torch.stack(batch_images, dim=0)

    return batch_images

In [14]:
images_names = ["/home/ubuntu/Shree_FYP/train/stage2/test/images/cat.png", "/home/ubuntu/Shree_FYP/train/stage2/test/images/dog.png"]

In [15]:
images_batches = [
    [images_names[0]],
    [images_names[1]],
]

In [16]:
unnorm_imgs = preprocess_raw_image_batches_for_vggt(
    image_batches=images_batches,
    mode="crop",
    device=device,
)

In [17]:
unnorm_imgs.shape

torch.Size([2, 1, 3, 518, 518])

In [18]:
with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=dtype):
        vggt_output = vggt_model(unnorm_imgs)

In [19]:
vggt_output["features"][-1].shape

torch.Size([2, 1, 1374, 2048])

In [20]:
agg_vggt_hidden = vggt_output["features"][-1] 

In [21]:
patch_start_idx = vggt_output["patch_start_idx"]

In [22]:
original_image = vggt_output["images"]

In [23]:
vggt_hidden = agg_vggt_hidden[:, :, patch_start_idx:, :]

In [24]:
print("vision_hidden:", vision_hidden.shape)
print("vggt_hidden before pooling:", vggt_hidden.shape)

vision_hidden: torch.Size([2, 1024, 2560])
vggt_hidden before pooling: torch.Size([2, 1, 1369, 2048])


In [25]:
import torch
import torch.nn as nn
import numpy as np
from typing import List, Dict, Tuple, Union
from einops import rearrange

from vggt.heads.utils import create_uv_grid, position_grid_to_embed


def _interpolate(
    x: torch.Tensor,
    size: Tuple[int, int] = None,
    scale_factor: float = None,
    mode: str = "bilinear",
    align_corners: bool = True,
) -> torch.Tensor:
    """
    Custom interpolate to avoid INT_MAX issues in nn.functional.interpolate.
    """
    if size is None:
        size = (int(x.shape[-2] * scale_factor), int(x.shape[-1] * scale_factor))

    INT_MAX = 1610612736

    input_elements = size[0] * size[1] * x.shape[0] * x.shape[1]

    if input_elements > INT_MAX:
        chunks = torch.chunk(x, chunks=(input_elements // INT_MAX) + 1, dim=0)
        interpolated_chunks = [
            nn.functional.interpolate(chunk, size=size, mode=mode, align_corners=align_corners) for chunk in chunks
        ]
        x = torch.cat(interpolated_chunks, dim=0)
        return x.contiguous()
    else:
        return nn.functional.interpolate(x, size=size, mode=mode, align_corners=align_corners)

def _apply_pos_embed(x: torch.Tensor, W: int, H: int, ratio: float = 0.1) -> torch.Tensor:
    """
    Apply positional embedding to tensor x.
    """
    patch_w = x.shape[-1]
    patch_h = x.shape[-2]
    pos_embed = create_uv_grid(patch_w, patch_h, aspect_ratio=W / H, dtype=x.dtype, device=x.device)
    pos_embed = position_grid_to_embed(pos_embed, x.shape[1])
    pos_embed = pos_embed * ratio
    pos_embed = pos_embed.permute(2, 0, 1)[None].expand(x.shape[0], -1, -1, -1)
    return x + pos_embed

def interpolate_pooling(hidden, patch_hw, img_hw, reference, pooling_func, use_vggt_pe):
    (patch_h, patch_w) = patch_hw
    (img_h, img_w) = img_hw
    bs, N, S, D = hidden.shape
    re_sample_ratio = 1 / np.sqrt(N * S / reference.shape[1])

    _hidden = hidden.permute(0, 1, 3, 2)
    _hidden = _hidden.reshape(bs*N, D, patch_h, patch_w)
    if use_vggt_pe:
        _hidden = _apply_pos_embed(_hidden, img_w, img_h)
    hidden_pooling = _interpolate(
        _hidden, scale_factor=re_sample_ratio, mode=pooling_func, align_corners=True
    )
    hidden_pooling = hidden_pooling.reshape(bs, N, D, -1).permute(0, 1, 3, 2).reshape(bs, -1, D)
    return hidden_pooling


def custom_pooling(hidden, patch_hw, img_hw, reference, pooling_func, use_vggt_pe):
    if pooling_func in ['bilinear']:
        return interpolate_pooling(hidden, patch_hw, img_hw, reference, pooling_func, use_vggt_pe)
    else:
        raise NotImplementedError(f"Pooling function {pooling_func} is not implemented.")


In [28]:
H, W = original_image.shape[-2:]
patch_h, patch_w = H // vggt_model.patch_size, W // vggt_model.patch_size

In [29]:
vggt_hidden = custom_pooling(vggt_hidden, (patch_h, patch_w), (H, W), vision_hidden, "bilinear", use_vggt_pe=False)

In [30]:
print("vggt_hidden after pooling:", vggt_hidden.shape)

vggt_hidden after pooling: torch.Size([2, 1024, 2048])


In [102]:
class AlignProjector(nn.Module):
    """
    Projects Qwen3.5 4B vision embeddings (2560) to VGGT embeddings (2048) 
    and computes the Cosine Alignment Loss.
    """
    def __init__(
            self, 
            llm_dim: int = 2560,     # Qwen3.5 4B hidden size
            vggt_dim: int = 2048,    # Your specific VGGT embed_dim
            align_loss_type: str = "cosine",
            use_vlm_norm: bool = False,
        ) -> None:
        super().__init__()
        self.llm_dim = llm_dim
        self.vggt_dim = vggt_dim
        self.align_loss_type = align_loss_type

        # Bottleneck layer to prevent overfitting and save VRAM
        # Max of (2048, 2560//2) = 2048
        hidden_dim = max(self.vggt_dim, self.llm_dim // 2) 
        
        self.fc1 = nn.Linear(self.llm_dim, hidden_dim, bias=True)
        self.fc2 = nn.Linear(hidden_dim, self.vggt_dim, bias=True) # Maps exactly to 2048
        self.act_fn1 = nn.GELU()
        
        self.vlm_norm = nn.LayerNorm(llm_dim) if use_vlm_norm else None
        self.initialize_weights()
    
    def initialize_weights(self):
        def _basic_init(module):
            if isinstance(module, nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
        self.apply(_basic_init)

    def align_dimension(self, LLM_embedding: torch.Tensor) -> torch.Tensor:
        if self.vlm_norm is not None:
            LLM_embedding = self.vlm_norm(LLM_embedding)
        projected_features = self.fc1(LLM_embedding)
        projected_features = self.act_fn1(projected_features)
        projected_features = self.fc2(projected_features)
        return projected_features
    
    def compute_align_loss_cosine(self, vision_hidden, vggt_hidden):
        align_loss = 0.0
        bsz = vision_hidden.shape[0]
        
        for _vision, _vggt in zip(vision_hidden, vggt_hidden):
            _vision = torch.nn.functional.normalize(_vision, dim=-1)
            _vggt = torch.nn.functional.normalize(_vggt, dim=-1)
            
            # Dot product over the feature dimension, then mean over the sequence length
            cosine_sim = (_vision * _vggt).sum(dim=-1)
            align_loss += 1.0 - cosine_sim.mean() 
            
        align_loss /= bsz
        return align_loss
    
    def forward(self, LLM_emb, target_emb):
        if self.align_loss_type == "cosine":
            # Project in bf16 to save VRAM
            with torch.autocast("cuda", dtype=torch.bfloat16):
                LLM_emb = self.align_dimension(LLM_emb)
                
            # CRITICAL: Cast to fp32 for cosine similarity to avoid underflow/NaNs in bf16
            align_loss = self.compute_align_loss_cosine(LLM_emb.float(), target_emb.float())
            return align_loss
        else:
            raise NotImplementedError(f"Align loss type {self.align_loss_type} is not implemented.")

In [103]:
align_projector = AlignProjector(
    llm_dim=2560,   # Qwen
    vggt_dim=2048,  # Your VGGT
    align_loss_type="cosine",
    use_vlm_norm=False 
).to(device)

In [109]:
with torch.amp.autocast("cuda", dtype = torch.bfloat16):
    align_loss = align_projector(vision_hidden, vggt_hidden.detach())

In [110]:
align_loss

tensor(1.0123, device='cuda:0', grad_fn=<DivBackward0>)